# Классификация веб-страниц: контент 18+ или нет

**Задача.** По `url` и `title` (заголовку) веб-страницы предсказать метку: `1` — контент 18+, `0` — нет. Это **бинарная классификация**, обучение с учителем.

**Данные** (не входят в репозиторий, выдаются в курсе):
- `train.csv` — 135 309 страниц: `ID`, `url`, `title`, `label`;
- `test.csv` — те же колонки без `label`; для них нужно предсказать ответ.

**Метрика — F1-score.** Среднее гармоническое точности (precision) и полноты (recall):

$$
\text{precision} = \frac{TP}{TP + FP} \qquad
\text{recall} = \frac{TP}{TP + FN} \qquad
F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}
$$

- **TP** — страница 18+, и модель сказала 18+;
- **FP** — обычная страница, а модель сказала 18+ (ложная тревога);
- **FN** — страница 18+, а модель её пропустила.

Почему F1, а не accuracy: страниц 18+ всего ~12%. Модель «всегда 0» дала бы accuracy 88% и не нашла бы ни одной страницы 18+. F1 так не обмануть: если precision или recall близки к нулю, F1 тоже близок к нулю.

**Формат ответа:** `ID,label`.

## Итог

| # | Решение | F1 (validation) |
|---|---|---|
| 1 | Правило «в заголовке есть слово порно» | 0.312 |
| 2 | CountVectorizer + наивный байес, только title | 0.910 |
| 3 | CountVectorizer + наивный байес, url + title | 0.930 |
| 4 | CountVectorizer + логистическая регрессия, url + title | 0.964 |
| 5 | то же + подобранный порог 0.3 | **0.970** |

Правило работы: **меняем одну вещь за раз** и смотрим, как изменился F1. Иначе непонятно, что помогло.

## 1. Загрузка данных

- `pandas` — работа с таблицами: чтение csv, колонки, сохранение ответа.
- `numpy` — работа с массивами чисел: понадобится для перебора порогов.

In [1]:
import pandas as pd   # таблицы
import numpy as np    # массивы чисел

In [2]:
train_df = pd.read_csv('train.csv')   # читаем обучающую выборку в таблицу (DataFrame)
train_df.head(10)                     # первые 10 строк — посмотреть глазами, что внутри

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0
5,5,beregifiguru.ru,Салат: корейская морковь + копченая курица + к...,0
6,6,ekb.vseinstrumenti.ru,"Угловой пневмогайковерт FUBAG RWC 105 1/2"" 100...",0
7,7,mirtabaka.org,Табачный магазин `Мир Табака` | КУРИТЕЛЬНЫЕ ТР...,0
8,8,xlecx.com,league of legends » Page 5 » Porn comics free ...,1
9,9,bus.biletyplus.ua,"08:00 (4766р), Одесса - Ростов-на-Дону 20 сент...",0


In [ ]:
test_df = pd.read_csv('test.csv')     # тестовая выборка: колонки label НЕТ — её мы и предсказываем
test_df.head(10)

> В `test` нет колонки `label`, поэтому **F1 на test посчитать нельзя** — это делает только сайт соревнования. Чтобы проверять себя по ходу работы, в шаге 5 отрежем кусок от train (validation).

## 2. Пропуски

Первый запуск правила из шага 3 упал с ошибкой `TypeError: argument of type 'float' is not iterable`.

**Почему.** Если клетка в csv пустая, pandas записывает в неё `NaN` («не число»), а `NaN` в Python имеет тип `float`. Код искал слово внутри строки, а получил число. Значит, где-то пустой заголовок.

**Как нашли.** `isna()` для каждой клетки проверяет, пустая ли она (True/False), `sum()` складывает True по колонкам — получаем число пропусков в каждой колонке:

In [ ]:
train_df.isna().sum()

ID       0
url      0
title    1
label    0
dtype: int64

Пустой заголовок ровно один. Заполняем пропуск **пустой строкой**: заголовка нет — значит, и слов нет. Среднее или медиана для текста не имеют смысла. Делаем одинаково для train и test.

In [4]:
from sklearn.metrics import f1_score, precision_score, recall_score   # метрики качества

In [9]:
train_df["title"] = train_df["title"].fillna("")   # NaN → пустая строка
test_df["title"] = test_df["title"].fillna("")

X_train = train_df["title"].values   # по чему предсказываем: заголовки
X_test = test_df["title"].values
y_train = train_df["label"].values   # что предсказываем: правильные ответы 0/1

## 3. Baseline: правило «есть слово порно»

**Baseline** — самое простое решение, точка отсчёта. Любая модель обязана его побить, иначе она бесполезна.

Правило: если в заголовке есть подстрока «порно» → `1`, иначе `0`.
- `"порно" in text` — True, если подстрока есть;
- `int(...)` — True → 1, False → 0;
- `[... for text in X_train]` — применить ко всем заголовкам и собрать список.

In [10]:
y_pred = [int("порно" in text) for text in X_train]
f1_score(y_train, y_pred)   # сначала правильные ответы, потом предсказания

0.3124275855120649

F1 = 0.31 — плохо. Чтобы понять **почему**, разложим F1 на precision и recall:

In [11]:
print("precision:", precision_score(y_train, y_pred))
print("recall:   ", recall_score(y_train, y_pred))

precision: 0.9888392857142857
recall:    0.1855219862399043


- **precision 0.99** — когда правило говорит «18+», оно почти всегда право;
- **recall 0.19** — но находит только 19% страниц 18+.

Проблема в **пропусках**: большинство таких страниц не пишут «порно» прямым текстом («xxx», «секс», «Porn» по-английски, «Отборная домашка»). А ещё правило ловит «опорно» в «Болезни опорно-двигательной системы» — это ложная тревога.

Вывод: нужно, чтобы решение **само нашло все подозрительные слова**. Это и делает ML-модель.

## 4. ML baseline: мешок слов + наивный байес

**Проблема:** модель понимает только числа, а у нас текст. Нужно **извлечение признаков** — превратить текст в вектор.

**`CountVectorizer` — мешок слов.** Каждое слово словаря становится столбцом, в клетке — сколько раз слово встретилось в тексте:

| | кино | онлайн | порно | смотреть |
|---|---|---|---|---|
| «смотреть кино онлайн» | 1 | 1 | 0 | 1 |
| «смотреть порно онлайн онлайн» | 0 | 2 | 1 | 1 |

Порядок слов теряется, важно только какие слова есть и сколько раз. В наших данных ~190 тысяч разных слов, но в одном заголовке их ~10, поэтому матрица почти вся из нулей — **разреженная**: хранятся только ненулевые клетки.

**`MultinomialNB` — наивный байес.** Для каждого слова модель считает, насколько чаще оно встречается в страницах 18+, чем в обычных, а для нового текста складывает эти «голоса» слов. «Наивный» — потому что считает слова независимыми друг от друга.

**Методы sklearn:**
- `fit_transform(X)` — **выучить словарь** (какие будут столбцы) и сразу заполнить таблицу;
- `fit(X, y)` — обучить модель;
- `predict(X)` — получить ответы 0/1.

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)   # текст → таблица слов

model = MultinomialNB()
model.fit(X_train_vectorized, y_train)                  # обучение
y_pred = model.predict(X_train_vectorized)              # предсказание

print("F1:       ", f1_score(y_train, y_pred))
print("precision:", precision_score(y_train, y_pred))
print("recall:   ", recall_score(y_train, y_pred))

F1:        0.9036641864382985
precision: 0.8514285714285714
recall:    0.9627280885432247


recall вырос с 0.19 до **0.96**: модель сама нашла тысячи «подозрительных» слов. Платой стал precision (0.99 → 0.85) — это компромисс precision ↔ recall.

> **Подвох:** модель училась на `X_train` и проверялась на нём же. Она уже видела эти заголовки вместе с ответами, поэтому оценка может быть завышенной. Исправляем в следующем шаге.

## 5. Честная проверка: validation

Отрезаем от train 20% и **прячем от модели**: на них проверяем качество.

| Часть | Доля | Зачем |
|---|---|---|
| train (`X_tr`) | 80% | учим модель |
| validation (`X_val`) | 20% | считаем F1 на том, чего модель не видела |

Параметры `train_test_split`:
- `test_size=0.2` — 20% уходит в validation;
- `stratify=y_train` — **стратификация**: доля страниц 18+ одинаковая в обеих частях, иначе F1 скакал бы от разбиения к разбиению;
- `random_state=42` — одинаковое разбиение при каждом запуске, чтобы результаты можно было сравнивать.

**Главное правило:** словарь (`fit`) учим только на обучающей части. На validation — только `transform`: перевести тексты в таблицу по **уже выученному** словарю. Если выучить словарь заново, столбцы перемешаются и модель будет читать не те слова.

In [13]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,       # 20% прячем для проверки
    stratify=y_train,    # доля 18+ сайтов одинаковая в обеих частях
    random_state=42,     # одинаковое разбиение при каждом запуске
)

vectorizer = CountVectorizer()
X_tr_vec = vectorizer.fit_transform(X_tr)    # словарь учим только на 80%
X_val_vec = vectorizer.transform(X_val)      # спрятанные 20% только переводим в таблицу

model = MultinomialNB()
model.fit(X_tr_vec, y_tr)                    # учимся только на 80%

print("F1 на том, что видела:   ", f1_score(y_tr, model.predict(X_tr_vec)))
print("F1 на том, что не видела:", f1_score(y_val, model.predict(X_val_vec)))

F1 на том, что видела:    0.9035454513510665
F1 на том, что не видела: 0.9101641232363951


На validation F1 даже чуть выше (0.910 против 0.904) — разница в пределах шума. Наивный байес — простая модель: он только считает частоты слов и не может «зазубрить» отдельные заголовки, поэтому **переобучения нет**.

Дальше все решения принимаем **только по F1 на validation**.

## 6. Добавляем url

Модель видела только заголовок, а в адресе часто ответ написан прямо: `national-porn.com`.

Самый простой способ — **склеить url и title в один текст** через пробел. `CountVectorizer` считает точки и дефисы разделителями, поэтому `national-porn.com` превратится в слова `national`, `porn`, `com`.

In [14]:
train_df["title"] = train_df["title"].fillna("")
test_df["title"] = test_df["title"].fillna("")

# новая колонка: url + пробел + title (без пробела последнее слово url слиплось бы с первым словом заголовка)
train_df["text"] = train_df["url"] + " " + train_df["title"]
test_df["text"] = test_df["url"] + " " + test_df["title"]

X_train = train_df["text"].values
X_test = test_df["text"].values
y_train = train_df["label"].values

In [15]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,       # 20% прячем для проверки
    stratify=y_train,    # доля 18+ сайтов одинаковая в обеих частях
    random_state=42,     # одинаковое разбиение при каждом запуске
)

vectorizer = CountVectorizer()
X_tr_vec = vectorizer.fit_transform(X_tr)    # словарь учим только на 80%
X_val_vec = vectorizer.transform(X_val)      # спрятанные 20% только переводим в таблицу

model = MultinomialNB()
model.fit(X_tr_vec, y_tr)                    # учимся только на 80%

print("F1 на том, что видела:   ", f1_score(y_tr, model.predict(X_tr_vec)))
print("F1 на том, что не видела:", f1_score(y_val, model.predict(X_val_vec)))

F1 на том, что видела:    0.924724342663274
F1 на том, что не видела: 0.9303060784595488


F1 на validation: **0.910 → 0.930**. url помог.

## 7. Логистическая регрессия

Признаки те же, меняем **только модель**.

Как работает логистическая регрессия:
1. Каждому слову — свой вес $w_j$: у «porn» большой положительный, у «новости» отрицательный.
2. Для страницы складываем веса её слов: $z = \langle x, w \rangle = w_1 x_1 + w_2 x_2 + \dots$
3. Пропускаем через **сигмоиду** — получаем вероятность 18+:
$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$
4. Веса подбираются минимизацией **log-loss** — штрафа за уверенные ошибки:
$$\text{LogLoss} = -\frac{1}{N}\sum_{i=1}^{N}\Big[y_i \log p_i + (1 - y_i)\log(1 - p_i)\Big]$$

Чем лучше наивного байеса: байес оценивает каждое слово отдельно, а логрегрессия подбирает все веса **вместе** и учитывает, что слова повторяют друг друга.

`max_iter=1000` — сколько шагов оптимизации разрешено сделать. Весов ~200 тысяч, стандартных 100 шагов может не хватить (тогда sklearn выдаёт `ConvergenceWarning`).

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

vectorizer = CountVectorizer()
X_tr_vec = vectorizer.fit_transform(X_tr)
X_val_vec = vectorizer.transform(X_val)

model = LogisticRegression(max_iter=1000)    # ← поменяли только модель
model.fit(X_tr_vec, y_tr)

print(model)   # проверка: какая модель на самом деле обучилась
print("F1 на том, что видела:   ", f1_score(y_tr, model.predict(X_tr_vec)))
print("F1 на том, что не видела:", f1_score(y_val, model.predict(X_val_vec)))

LogisticRegression(max_iter=1000)
F1 на том, что видела:    0.9904560715228791
F1 на том, что не видела: 0.9644779332615716


- validation: **0.930 → 0.964**;
- появился **разрыв**: 0.990 на train против 0.964 на validation. Это **переобучение**: у модели по весу на каждое из ~200 тысяч слов, и часть весов подогналась под конкретные заголовки train. Лечится регуляризацией — следующий шаг улучшений.

> По дороге была ошибка: после замены модели F1 вышел точь-в-точь прежним. `print(model)` показал `MultinomialNB()` — новая строка не применилась. Поэтому `print(model)` теперь стоит в ячейке.

## 8. Подбор порога

`predict` ставит `1`, если вероятность больше **0.5**. Но 0.5 ничем не особенный. Порог двигает компромисс:
- **ниже** порог → больше страниц считаем 18+ → recall растёт, precision падает;
- **выше** — наоборот.

Ищем порог с максимальным F1 перебором на validation:
- `predict_proba(X)` — вероятности классов; `[:, 1]` — столбец вероятности класса 1 (18+);
- `np.arange(0.1, 0.9, 0.05)` — пороги 0.10, 0.15, …, 0.85;
- `(proba_val > t).astype(int)` — сравнить вероятности с порогом и превратить True/False в 1/0.

In [20]:
import numpy as np

proba_val = model.predict_proba(X_val_vec)[:, 1]   # вероятность 18+ для каждого сайта

for t in np.arange(0.1, 0.9, 0.05):                # пробуем пороги 0.10, 0.15, ..., 0.85
    y_pred_t = (proba_val > t).astype(int)         # больше порога → 1, иначе 0
    print(f"порог {t:.2f}   F1 = {f1_score(y_val, y_pred_t):.4f}")

порог 0.10   F1 = 0.9568
порог 0.15   F1 = 0.9672
порог 0.20   F1 = 0.9693
порог 0.25   F1 = 0.9706
порог 0.30   F1 = 0.9699
порог 0.35   F1 = 0.9699
порог 0.40   F1 = 0.9686
порог 0.45   F1 = 0.9660
порог 0.50   F1 = 0.9645
порог 0.55   F1 = 0.9623
порог 0.60   F1 = 0.9594
порог 0.65   F1 = 0.9544
порог 0.70   F1 = 0.9515
порог 0.75   F1 = 0.9472
порог 0.80   F1 = 0.9409
порог 0.85   F1 = 0.9304


- при пороге 0.50 F1 = 0.9645, лучший — **0.25** (F1 = 0.9706);
- от 0.25 до 0.35 F1 почти не меняется — это **плато**. Берём его середину, **0.3**: порог подобран на validation и немного под неё подогнан, а в середине плато небольшой сдвиг на тесте не сбросит нас со склона;
- лучший порог ниже 0.5, потому что страниц 18+ меньшинство и модель осторожничает: даже на настоящих 18+ часто даёт вероятность 0.3–0.4.

+0.006 к F1 почти даром: модель не меняли, сдвинули только границу.

## 9. Финальная модель и файл ответов

Всё решили (модель, порог), поэтому validation больше не прячем: учим на **100% train** — больше данных, модель чуть лучше.

- `vectorizer_full.transform(X_test)` — test только переводим по словарю train: столбцы должны совпадать;
- `to_csv(..., index=False)` — без лишней колонки с номерами строк, иначе формат не совпадёт с заданием;
- **проверка здравого смысла:** доля 18+ в ответах должна быть близка к доле в train. Если там 0% или 50% — что-то сломалось.

In [21]:
# 1. Учим словарь и модель на ВСЁМ train
vectorizer_full = CountVectorizer()
X_train_vec = vectorizer_full.fit_transform(X_train)
X_test_vec = vectorizer_full.transform(X_test)          # test только переводим в таблицу

model_full = LogisticRegression(max_iter=1000)
model_full.fit(X_train_vec, y_train)

# 2. Вероятности для test и порог
proba_test = model_full.predict_proba(X_test_vec)[:, 1]
test_df["label"] = (proba_test > 0.3).astype(int)

# 3. Файл ответов в формате из задания: ID,label
test_df[["ID", "label"]].to_csv("submission_logreg.csv", index=False)

print(test_df[["ID", "label"]].head())
print("доля 18+ в ответах:", test_df["label"].mean())
print("доля 18+ в train:  ", y_train.mean())

       ID  label
0  135309      0
1  135310      0
2  135311      0
3  135312      1
4  135313      0
доля 18+ в ответах: 0.12079599463048289
доля 18+ в train:   0.12353206364691188


Доли совпадают: 12.1% в ответах против 12.4% в train — файл выглядит правильным.

## Что дальше

| Идея | Зачем |
|---|---|
| Регуляризация (параметр `C` в `LogisticRegression`) | уменьшить разрыв train/validation |
| `TfidfVectorizer` вместо `CountVectorizer` | приглушить частые бесполезные слова (`www`, `com`, `ru`) |
| Символьные n-граммы для url | url часто слеплен: `freexxxvideos` по словам не разрезать |
| Отдельные векторайзеры для url и title | слово в адресе и в заголовке может значить разное |
| Переподбор порога после каждого изменения | лучший порог меняется вместе с моделью |